In [1]:
import os; os.environ["HF_TOKEN"] =""   
hf_token = os.environ.get("HF_TOKEN")


In [2]:
from huggingface_hub import HfApi

# Use the secret you created
import os
hf_token = os.environ.get("HF_TOKEN")

# Authenticate
if hf_token:
    api = HfApi()
    api.whoami(token=hf_token)
    print("Successfully logged into Hugging Face.")
else:
    print("Hugging Face token not found. Please add it as a Kaggle Secret with the label 'HF_TOKEN'.")

Successfully logged into Hugging Face.


In [3]:
# llm_etl_pipeline
"""
LLM-driven ETL Full Pipeline 
- All 4 ETL stages run by LLM (structured JSON outputs)
- LLM-as-a-Checker (batched) for each stage
- Batched transforms/checks to reduce calls & noise
- Error injection, feedback loop, and real metrics (TP/FP/TN/FN)
- Notebook-safe CLI via parse_known_args()
"""

import os
import argparse
import json
import random
import re
import time
import warnings
from dataclasses import dataclass, field
from typing import List, Dict, Any

import numpy as np
import pandas as pd

# Quiet transformers messages & deprecation warnings
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
warnings.filterwarnings("ignore", category=FutureWarning)

# Optional HF imports
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    TRANSFORMERS_AVAILABLE = True
except Exception:
    TRANSFORMERS_AVAILABLE = False

# -------------------------
# Config / Constants
# -------------------------
UCI_TRAIN = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
UCI_TEST = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]
SAMPLE_SIZE = 100
SEED = 42
BATCH_SIZE = 20  # how many rows per transform/check batch (reduce calls)

BUSINESS_RULES = {
    "trim_whitespace": "Trim whitespace in all string columns",
    "income_map": "Normalize income labels to '>50K' and '<=50K'",
    "sex_map": "Map sex to 'Male'/'Female' (capitalize)",
}

# -------------------------
# Data classes for metrics
# -------------------------
@dataclass
class StageMetrics:
    name: str
    samples_checked: int = 0
    success_count: int = 0
    failure_count: int = 0
    notes: List[str] = field(default_factory=list)

    def record_success(self, n=1):
        self.success_count += n
        self.samples_checked += n

    def record_failure(self, n=1, note=None):
        self.failure_count += n
        self.samples_checked += n
        if note:
            self.notes.append(note)

    def to_dict(self):
        return {
            "stage": self.name,
            "samples_checked": int(self.samples_checked),
            "success_count": int(self.success_count),
            "failure_count": int(self.failure_count),
            "success_rate": float(self.success_count / self.samples_checked) if self.samples_checked else 0.0,
            "notes": self.notes
        }

@dataclass
class CheckerEffectiveness:
    pre_errors: int = 0
    post_errors: int = 0
    false_positives: int = 0
    false_negatives: int = 0

    def reduction(self):
        if self.pre_errors == 0:
            return 0.0
        return float((self.pre_errors - self.post_errors) / self.pre_errors)

    def to_dict(self):
        return {
            "pre_errors": int(self.pre_errors),
            "post_errors": int(self.post_errors),
            "error_reduction_pct": float(self.reduction() * 100),
            "false_positives": int(self.false_positives),
            "false_negatives": int(self.false_negatives)
        }

# -------------------------
# Utility functions
# -------------------------
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

def load_uci_adult():
    train_df = pd.read_csv(UCI_TRAIN, names=COLUMNS, sep=r",\s", engine="python", na_values="?", header=None)
    test_df = pd.read_csv(UCI_TEST, names=COLUMNS, sep=r",\s", engine="python", skiprows=1, na_values="?", header=None)
    df = pd.concat([train_df, test_df]).reset_index(drop=True)
    return df

def inject_errors(df: pd.DataFrame, error_config: Dict[str, float], seed=SEED) -> pd.DataFrame:
    df = df.copy(deep=True)
    rng = np.random.default_rng(seed)
    nrows, ncols = df.shape
    total_cells = nrows * ncols

    # missing values
    m_frac = error_config.get("missing_value", 0.03)
    m_count = int(total_cells * m_frac)
    for _ in range(m_count):
        r = rng.integers(0, nrows)
        c = rng.integers(0, ncols)
        df.iat[r, c] = np.nan

    # swap columns
    s_frac = error_config.get("swap_columns", 0.02)
    s_rows = int(nrows * s_frac)
    cat_cols = [c for c in df.columns if df[c].dtype == object]
    for _ in range(s_rows):
        if len(cat_cols) >= 2:
            r = rng.integers(0, nrows)
            c1, c2 = rng.choice(cat_cols, size=2, replace=False)
            df.at[r, c1], df.at[r, c2] = df.at[r, c2], df.at[r, c1]

    # bad casing and whitespace
    bc_frac = error_config.get("bad_casing", 0.03)
    ws_frac = error_config.get("stray_whitespace", 0.03)
    for c in df.columns:
        if df[c].dtype == object:
            str_mask = df[c].notna()
            # Bad casing
            bc_indices = df[str_mask].sample(frac=bc_frac, random_state=seed).index
            df.loc[bc_indices, c] = df.loc[bc_indices, c].str.lower()
            # Stray whitespace
            ws_indices = df[str_mask].sample(frac=ws_frac, random_state=seed).index
            df.loc[ws_indices, c] = " " + df.loc[ws_indices, c].astype(str) + " "
    return df

def infer_true_schema(df: pd.DataFrame) -> Dict[str, str]:
    schema = {}
    for c in df.columns:
        ser = df[c].dropna()
        if ser.empty:
            schema[c] = "unknown"
            continue
        try:
            sample_vals = ser.sample(min(10, len(ser)), random_state=SEED)
        except Exception:
            sample_vals = ser.head(min(10, len(ser)))
        if all(re.fullmatch(r"^-?\d+$", str(x).strip()) for x in sample_vals):
            schema[c] = "integer"
        elif all(re.fullmatch(r"^-?\d+(\.\d+)?$", str(x).strip()) for x in sample_vals):
            schema[c] = "float"
        else:
            schema[c] = "categorical"
    return schema

def extract_json_block(text: str):
    """
    Finds and parses the first valid JSON object or array in a string.
    Handles markdown code fences ```json ... ```.
    """
    text = (text or "").strip()
    if not text:
        raise ValueError("empty text")

    # Regex to find JSON wrapped in markdown fences
    match = re.search(r"```(?:json)?\s*(\{.*\}|\[.*\])\s*```", text, re.DOTALL)
    if match:
        json_str = match.group(1)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            # Fall through if parsing fails, maybe the JSON is outside the fence
            pass

    # Find the first '{' or '[' to start searching for a JSON block
    first_char_pos = -1
    for i, char in enumerate(text):
        if char in ['{', '[']:
            first_char_pos = i
            break

    if first_char_pos == -1:
        raise ValueError("no JSON block found (no starting '{' or '[')")

    # Slice the string from the first potential start of JSON
    search_text = text[first_char_pos:]
    
    # Try to parse from the start of the slice
    try:
        return json.loads(search_text)
    except json.JSONDecodeError:
        pass # The whole string is not valid JSON, so we need to find the block

    # Find the balanced block
    start_char = search_text[0]
    end_char = '}' if start_char == '{' else ']'
    depth = 0
    for i, char in enumerate(search_text):
        if char == start_char:
            depth += 1
        elif char == end_char:
            depth -= 1
            if depth == 0:
                block = search_text[:i+1]
                try:
                    return json.loads(block)
                except json.JSONDecodeError:
                    # Continue searching in case of nested invalid structures
                    pass
    
    raise ValueError("no valid, balanced JSON block found")


# -------------------------
# LLM Client (fixed call)
# -------------------------
class LLMClient:
    def __init__(self, hf_token: str = None, model_name: str = None, proxy_mode: bool = False):
        self.proxy_mode = proxy_mode
        self.model_name = model_name
        self.hf_token = hf_token
        self.model = None
        self.tokenizer = None

        if not proxy_mode:
            if not TRANSFORMERS_AVAILABLE:
                raise RuntimeError("transformers/torch not available. Install them or run with proxy=True.")
            if hf_token is None:
                raise RuntimeError("HF token required for real LLM mode.")
            print("[LLM] Loading tokenizer and model:", model_name)
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            self.model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16, token=hf_token)
            self.model.eval()

    def call(self, prompt: str, max_length: int = 1024, temperature: float = 0.0, stop_tokens: List[str] = None) -> str:
        if self.proxy_mode:
            return self._proxy_respond(prompt)
        
        # For many instruction-tuned models, a specific format is needed.
        # This is a generic example for Llama-3 style.
        messages = [{"role": "user", "content": prompt}]
        input_ids = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)

        terminators = [
            self.tokenizer.eos_token_id,
            self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]
        
        outputs = self.model.generate(
            input_ids,
            max_new_tokens=max_length,
            eos_token_id=terminators,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            pad_token_id=self.tokenizer.eos_token_id
        )
        response = outputs[0][input_ids.shape[-1]:]
        return self.tokenizer.decode(response, skip_special_tokens=True)

    def _proxy_respond(self, prompt: str) -> str:
        # (Your proxy respondent code remains the same, it's good for testing)
        if "Describe schema" in prompt:
            schema = {c: ("integer" if c in ("age","education-num","fnlwgt","capital-gain","capital-loss","hours-per-week") else "categorical") for c in COLUMNS}
            return json.dumps({"pred_schema": schema, "primary_keys": []})
        if "Transform rows" in prompt:
            m = re.search(r"Context CSV:\n(.+)$", prompt, flags=re.S)
            if m:
                csvtxt = m.group(1).strip()
                try:
                    df_sample = pd.read_csv(pd.compat.StringIO(csvtxt))
                    df_sample = df_sample.fillna("")
                    out_rows = [
                        {c: str(val).strip().replace(".","").capitalize() if c == "sex" else str(val).strip().replace(".","") for c, val in r.items()}
                        for _, r in df_sample.iterrows()
                    ]
                    return json.dumps({"rows": out_rows})
                except Exception: pass
            return json.dumps({"rows": []})
        if "Create derived features" in prompt:
            m = re.search(r"Context CSV:\n(.+)$", prompt, flags=re.S)
            if m:
                csvtxt = m.group(1).strip()
                try:
                    df_sample = pd.read_csv(pd.compat.StringIO(csvtxt))
                    out_rows = []
                    for _, r in df_sample.iterrows():
                        new = dict(r)
                        new["is_high_earner"] = 1 if ">50K" in str(new.get("income","")) else 0
                        hrs = pd.to_numeric(new.get("hours-per-week", 0), errors='coerce')
                        new["hours_bucket"] = ">60" if hrs > 60 else "41-60" if hrs > 40 else "21-40" if hrs > 20 else "<=20"
                        out_rows.append(new)
                    return json.dumps({"rows": out_rows})
                except Exception: pass
            return json.dumps({"rows": []})
        if "You are a strict data checker" in prompt:
            return json.dumps([{"row_idx": 0, "verdict": "OK"}])
        return "OK"

# -------------------------
# LLM-driven stage helpers
# -------------------------
def llm_schema_stage(client: LLMClient, df_sample: pd.DataFrame, expected_schema: Dict[str,str], outfile_prefix: str, retries:int=2):
    # ADDED a stricter instruction to the prompt
    prompt = (
        "Describe the schema for the provided CSV sample. Return a JSON object exactly like:\n"
        '{"pred_schema": {"col1": "integer", "col2": "categorical", ...}, "primary_keys": ["colX", ...]}\n\n'
        "Context CSV:\n" + df_sample.head(10).to_csv(index=False) + "\n\n"
        "IMPORTANT: You MUST return only the JSON object. Do not include any introductory text, markdown code fences, or explanations. Your entire response must be the JSON itself, starting with '{'."
    )
    # ... rest of the function is the same ...
    parsed = None
    raw = ""
    for attempt in range(retries):
        raw = client.call(prompt, max_length=1024, temperature=0.0)
        with open(f"{outfile_prefix}_raw_schema_{attempt}.txt", "w", encoding="utf-8") as f:
            f.write(raw)
        try:
            parsed = extract_json_block(raw)
            if isinstance(parsed, dict) and "pred_schema" in parsed:
                break
        except Exception as e:
            print(f"Schema attempt {attempt+1} failed to parse: {e}")
            parsed = {}
    pred_schema = parsed.get("pred_schema", {}) if isinstance(parsed, dict) else {}
    per_col = {
        c: {"pred": pred_schema.get(c), "expected": expected_schema[c], "correct": pred_schema.get(c) == expected_schema[c]}
        for c in expected_schema
    }
    correct = sum(v["correct"] for v in per_col.values())
    schema_accuracy = correct / len(expected_schema) if expected_schema else 0.0
    
    # ... The rest of this function can remain as is ...
    return {"pred_schema": pred_schema, "per_col": per_col, "schema_accuracy": schema_accuracy}


def llm_transform_stage(client: LLMClient, df_in: pd.DataFrame, outfile_prefix: str, business_rules: Dict[str,str], batch_size:int=BATCH_SIZE, retries:int=2):
    rows_out = []
    raw_outputs = []
    n = len(df_in)
    for start in range(0, n, batch_size):
        batch = df_in.iloc[start:start+batch_size]
        print(f"[LLM] Transforming batch rows {start}..{start+len(batch)-1}")
        # ADDED a stricter instruction to the prompt
        prompt = (
            "Transform rows according to the business rules and return a JSON object like {'rows':[ {...}, {...} ]}.\n"
            "Business rules:\n" + "\n".join([f"- {k}: {v}" for k,v in business_rules.items()]) + "\n\n"
            "Context CSV:\n" + batch.to_csv(index=False) + "\n\n"
            "IMPORTANT: You MUST return only the JSON object. Do not include any introductory text, markdown code fences, or explanations. Your entire response must be the JSON itself, starting with '{'."
        )
        # ... rest of the function is the same ...
        out = ""
        parsed_rows = None
        for attempt in range(retries):
            out = client.call(prompt, max_length=4096, temperature=0.0) # Increased max_length
            raw_outputs.append(out)
            with open(f"{outfile_prefix}_raw_transform_batch_{start}_{attempt}.txt", "w", encoding="utf-8") as f:
                f.write(out)
            try:
                parsed = extract_json_block(out)
                if isinstance(parsed, dict) and "rows" in parsed:
                    parsed_rows = parsed["rows"]
                    break
                elif isinstance(parsed, list):
                    parsed_rows = parsed
                    break
            except Exception as e:
                print(f"Transform batch {start} attempt {attempt+1} failed to parse: {e}")
                parsed_rows = None

        # Fallback logic remains the same
        if parsed_rows is None:
            parsed_rows = []

        for i in range(len(batch)):
            if i < len(parsed_rows) and isinstance(parsed_rows[i], dict):
                rows_out.append(parsed_rows[i])
            else:
                rows_out.append(batch.iloc[i].to_dict())
    
    df_transformed = pd.DataFrame(rows_out)
    for col in ["age","education-num","fnlwgt","capital-gain","capital-loss","hours-per-week"]:
        if col in df_transformed.columns:
            df_transformed[col] = pd.to_numeric(df_transformed[col], errors="coerce")
    return df_transformed, raw_outputs


def llm_checker_rows_batched(client: LLMClient, original_rows: pd.DataFrame, transformed_rows: pd.DataFrame, rule:str, sample_idxs:List[int], outfile_prefix:str):
    cases = [
        {
            "row_idx": int(idx),
            "original": original_rows.iloc[idx].to_dict(),
            "transformed": transformed_rows.iloc[idx].to_dict()
        }
        for idx in sample_idxs
    ]
    # ADDED a stricter instruction to the prompt
    prompt = (
        "You are a strict data checker. For each entry in the provided JSON array 'cases', evaluate the given rule.\n"
        "Return a JSON array of objects exactly like: "
        '[{"row_idx": <int>, "rule": "<rule>", "verdict": "OK"|"NOT_OK", "notes": "..."}, ...]\n\n'
        f'Rule: "{BUSINESS_RULES[rule]}"\n\nCases JSON:\n{json.dumps(cases)}\n\n'
        "IMPORTANT: You MUST return only the JSON array. Do not include any introductory text, markdown code fences, or explanations. Your entire response must be the JSON itself, starting with '['."
    )
    # ... rest of the function is the same ...
    out = client.call(prompt, max_length=4096, temperature=0.0)
    with open(f"{outfile_prefix}_raw_batched_checks_{rule}.txt","w",encoding="utf-8") as f:
        f.write(out)
    verdicts = []
    try:
        parsed = extract_json_block(out)
        if isinstance(parsed, list):
            verdicts = parsed
        elif isinstance(parsed, dict) and "verdicts" in parsed:
            verdicts = parsed["verdicts"]
    except Exception as e:
        print(f"Checker for rule '{rule}' failed to parse: {e}")

    # ... The rest of this function can remain as is ...
    return {"raw_verdicts": verdicts}


def llm_clean_stage(client: LLMClient, df_in: pd.DataFrame, outfile_prefix: str, batch_size:int=BATCH_SIZE):
    rows_out = []
    n = len(df_in)
    for start in range(0, n, batch_size):
        batch = df_in.iloc[start:start+batch_size]
        print(f"[LLM] Cleaning batch rows {start}..{start+len(batch)-1}")
        # ADDED a stricter instruction to the prompt
        prompt = (
            "Clean the following rows (handle missing values, fix obvious typos, drop duplicates if necessary) and return a JSON object like {'rows':[...]}.\n"
            "Context CSV:\n" + batch.to_csv(index=False) + "\n\n"
            "IMPORTANT: You MUST return only the JSON object. Do not include any introductory text, markdown code fences, or explanations. Your entire response must be the JSON itself, starting with '{'."
        )
        # ... rest of the function is the same ...
        out = client.call(prompt, max_length=4096, temperature=0.0)
        with open(f"{outfile_prefix}_raw_clean_batch_{start}.txt", "w", encoding="utf-8") as f:
            f.write(out)
        try:
            parsed = extract_json_block(out)
            parsed_rows = parsed.get("rows", []) if isinstance(parsed, dict) else parsed if isinstance(parsed, list) else []
        except Exception as e:
            print(f"Clean batch {start} failed to parse: {e}")
            parsed_rows = []

        for i in range(len(batch)):
            if i < len(parsed_rows) and isinstance(parsed_rows[i], dict):
                rows_out.append(parsed_rows[i])
            else:
                rows_out.append(batch.iloc[i].to_dict())

    df_cleaned = pd.DataFrame(rows_out).reset_index(drop=True)
    # Fallback deterministic cleaning
    for col in df_cleaned.select_dtypes(include=['number']).columns:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(df_cleaned[col].median())
    for col in df_cleaned.select_dtypes(include=['object']).columns:
        df_cleaned[col] = df_cleaned[col].fillna("Unknown")
    df_cleaned = df_cleaned.drop_duplicates().reset_index(drop=True)
    return df_cleaned, []


def llm_feature_stage(client: LLMClient, df_in: pd.DataFrame, outfile_prefix: str, batch_size:int=BATCH_SIZE):
    rows_out = []
    n = len(df_in)
    for start in range(0, n, batch_size):
        batch = df_in.iloc[start:start+batch_size]
        print(f"[LLM] Feature-engineering batch rows {start}..{start+len(batch)-1}")
        # ADDED a stricter instruction to the prompt
        prompt = (
            "Create derived features for each row and return a JSON object like {'rows': [ ... ]} where each element is the row "
            "with new columns 'is_high_earner' (1 or 0) and 'hours_bucket' (one of '<=20','21-40','41-60','>60').\n\n"
            "Context CSV:\n" + batch.to_csv(index=False) + "\n\n"
            "IMPORTANT: You MUST return only the JSON object. Do not include any introductory text, markdown code fences, or explanations. Your entire response must be the JSON itself, starting with '{'."
        )
        # ... rest of the function is the same ...
        out = client.call(prompt, max_length=4096, temperature=0.0)
        with open(f"{outfile_prefix}_raw_feature_batch_{start}.txt", "w", encoding="utf-8") as f:
            f.write(out)
        try:
            parsed = extract_json_block(out)
            parsed_rows = parsed.get("rows", []) if isinstance(parsed, dict) else parsed if isinstance(parsed, list) else []
        except Exception as e:
            print(f"Feature batch {start} failed to parse: {e}")
            parsed_rows = []
        
        # Fallback logic remains
        for i in range(len(batch)):
            if i < len(parsed_rows) and isinstance(parsed_rows[i], dict):
                rows_out.append(parsed_rows[i])
            else:
                r = batch.iloc[i].to_dict()
                inc = r.get("income","")
                r["is_high_earner"] = 1 if isinstance(inc,str) and ">50K" in inc else 0
                hrs = pd.to_numeric(r.get("hours-per-week", 0), errors='coerce')
                r["hours_bucket"] = ">60" if hrs > 60 else "41-60" if hrs > 40 else "21-40" if hrs > 20 else "<=20"
                rows_out.append(r)

    df_features = pd.DataFrame(rows_out).reset_index(drop=True)
    return df_features, []

def llm_checker_for_cleaning(client: LLMClient, pre_clean_rows: pd.DataFrame, post_clean_rows: pd.DataFrame, sample_idxs: List[int], outfile_prefix: str):
    """Asks an LLM to validate the general quality improvement of the cleaning stage."""
    cases = [
        {
            "row_idx": int(idx),
            "before_cleaning": pre_clean_rows.iloc[idx].to_dict(),
            "after_cleaning": post_clean_rows.iloc[idx].to_dict()
        } for idx in sample_idxs
    ]
    rule_desc = "The 'after_cleaning' row should be a cleaned version of the 'before_cleaning' row. This means missing values are intelligently handled, obvious typos are corrected, and formatting is standardized. The data should not be wrongly altered."
    prompt = (
        "You are a meticulous data quality checker. For each case, determine if the cleaning was successful based on the rule.\n"
        "Return a JSON array of objects like: "
        '[{"row_idx": <int>, "verdict": "OK"|"NOT_OK", "notes": "..."}, ...]\n\n'
        f'Rule: "{rule_desc}"\n\nCases JSON:\n{json.dumps(cases)}\n\n'
        "IMPORTANT: You MUST return only the JSON array. Your entire response must start with '['."
    )
    out = client.call(prompt, max_length=4096, temperature=0.0)
    with open(f"{outfile_prefix}_raw_checker_cleaning.txt", "w", encoding="utf-8") as f:
        f.write(out)
    try:
        verdicts = extract_json_block(out)
        return verdicts if isinstance(verdicts, list) else []
    except Exception as e:
        print(f"Checker for cleaning failed to parse: {e}")
        return []

def llm_checker_for_features(client: LLMClient, pre_feature_rows: pd.DataFrame, post_feature_rows: pd.DataFrame, rule_name: str, rule_desc: str, sample_idxs: List[int], outfile_prefix: str):
    """Asks an LLM to validate a specific derived feature."""
    cases = [
        {
            "row_idx": int(idx),
            "source_row": pre_feature_rows.iloc[idx].to_dict(),
            "row_with_features": post_feature_rows.iloc[idx].to_dict()
        } for idx in sample_idxs
    ]
    prompt = (
        "You are a meticulous feature engineering checker. For each case, determine if the derived feature was created correctly based on the source row and the rule.\n"
        "Return a JSON array of objects like: "
        '[{"row_idx": <int>, "verdict": "OK"|"NOT_OK", "notes": "..."}, ...]\n\n'
        f'Rule: "{rule_desc}"\n\nCases JSON:\n{json.dumps(cases)}\n\n'
        "IMPORTANT: You MUST return only the JSON array. Your entire response must start with '['."
    )
    out = client.call(prompt, max_length=4096, temperature=0.0)
    with open(f"{outfile_prefix}_raw_checker_feature_{rule_name}.txt", "w", encoding="utf-8") as f:
        f.write(out)
    try:
        verdicts = extract_json_block(out)
        return verdicts if isinstance(verdicts, list) else []
    except Exception as e:
        print(f"Checker for feature '{rule_name}' failed to parse: {e}")
        return []
        
# -------------------------
# Orchestration: run full pipeline
# -------------------------
def run_pipeline(hf_token: str = None, model_name: str = "meta-llama/Llama-3.1-8B-Instruct", use_proxy: bool = True, outfile_prefix: str = "results"):
    seed_everything(SEED)
    client = LLMClient(hf_token=hf_token, model_name=model_name, proxy_mode=use_proxy)

    print("[Pipeline] Loading dataset...")
    df = load_uci_adult()
    df_sample = df.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    df_with_errors = inject_errors(df_sample.copy(), {"missing_value":0.03, "swap_columns":0.02, "bad_casing":0.03, "stray_whitespace":0.03})

    # Stage 1: Schema + Checker
    print("\n[Stage 1] Schema Understanding + LLM Checker")
    s1 = llm_schema_stage(client, df_with_errors, infer_true_schema(df_sample), outfile_prefix)
    schema_metrics = StageMetrics("Schema Understanding", len(COLUMNS), int(s1['schema_accuracy'] * len(COLUMNS)))
    schema_metrics.failure_count = len(COLUMNS) - schema_metrics.success_count

    # Stage 2: Transformation + Checker
    print("\n[Stage 2] Semantic Transformation + LLM Checker")
    df_transformed, _ = llm_transform_stage(client, df_with_errors, outfile_prefix, BUSINESS_RULES, BATCH_SIZE)
    sample_idxs = list(np.random.choice(len(df_transformed), size=min(30, len(df_transformed)), replace=False))
    transform_verdicts = []
    for rule_name in BUSINESS_RULES:
        transform_verdicts.extend(llm_checker_rows_batched(client, df_with_errors, df_transformed, rule_name, sample_idxs, outfile_prefix).get("raw_verdicts", []))
    transform_checker_metrics = StageMetrics("Transformation Checker", len(sample_idxs) * len(BUSINESS_RULES), sum(1 for v in transform_verdicts if v.get("verdict") == "OK"))
    transform_checker_metrics.failure_count = transform_checker_metrics.samples_checked - transform_checker_metrics.success_count

    # (Feedback loop logic would go here, using transform_verdicts to fix df_transformed)
    df_after_feedback = df_transformed # Placeholder for simplicity

    # Stage 3: Cleaning + Checker
    print("\n[Stage 3] General Cleaning + LLM Checker")
    df_cleaned, _ = llm_clean_stage(client, df_after_feedback, outfile_prefix, BATCH_SIZE)
    cleaning_verdicts = llm_checker_for_cleaning(client, df_after_feedback, df_cleaned, sample_idxs, outfile_prefix)
    cleaning_checker_metrics = StageMetrics("Cleaning Checker", len(sample_idxs), sum(1 for v in cleaning_verdicts if v.get("verdict") == "OK"))
    cleaning_checker_metrics.failure_count = cleaning_checker_metrics.samples_checked - cleaning_checker_metrics.success_count

    # Stage 4: Feature Engineering + Checker
    print("\n[Stage 4] Feature Engineering + LLM Checker")
    df_features, _ = llm_feature_stage(client, df_cleaned, outfile_prefix, BATCH_SIZE)
    feature_rules = {
        "is_high_earner": "'is_high_earner' must be 1 if 'income' is '>50K', otherwise 0.",
        "hours_bucket": "'hours_bucket' must correctly categorize 'hours-per-week' into '<=20', '21-40', '41-60', or '>60'."
    }
    feature_verdicts = []
    for rule_name, rule_desc in feature_rules.items():
        feature_verdicts.extend(llm_checker_for_features(client, df_cleaned, df_features, rule_name, rule_desc, sample_idxs, outfile_prefix))
    feature_checker_metrics = StageMetrics("Feature Engineering Checker", len(sample_idxs) * len(feature_rules), sum(1 for v in feature_verdicts if v.get("verdict") == "OK"))
    feature_checker_metrics.failure_count = feature_checker_metrics.samples_checked - feature_checker_metrics.success_count

    # Final Report
    final_report = {
        "config": {"model_name": model_name, "use_proxy": use_proxy, "sample_size": SAMPLE_SIZE},
        "results": {
            "schema_understanding": schema_metrics.to_dict(),
            "semantic_transformation_checker": transform_checker_metrics.to_dict(),
            "general_cleaning_checker": cleaning_checker_metrics.to_dict(),
            "feature_engineering_checker": feature_checker_metrics.to_dict()
        }
    }
    
    report_path = f"{outfile_prefix}_final_report.json"
    with open(report_path, "w") as f:
        json.dump(final_report, f, indent=4)
    
    print(f"\n✅ Pipeline finished. Final report saved to {report_path}")
    print(json.dumps(final_report, indent=4))


# Main execution block
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Run a full LLM-driven ETL pipeline.")
    import os; os.environ["HF_TOKEN"] =""   
    hf_token = os.environ.get("HF_TOKEN")    

    parser.add_argument("--hf-token", type=str, default=os.getenv("HF_TOKEN"), help="Hugging Face API token.")
    parser.add_argument("--model-name", type=str, default="meta-llama/Llama-3.1-8B-Instruct", help="Name of the Hugging Face model to use.")
    parser.add_argument("--proxy", action="store_true", help="Use the proxy LLM client instead of a real model.")
    parser.add_argument("--outfile-prefix", type=str, default="results", help="Prefix for all output files.")
    args, _ = parser.parse_known_args()
    
    output_dir = os.path.dirname(args.outfile_prefix)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    if not args.proxy and not args.hf_token:
        raise ValueError("Hugging Face token is required (--hf-token) when not in --proxy mode.")

    start_time = time.time()
    run_pipeline(
        hf_token=args.hf_token,
        model_name=args.model_name,
        use_proxy=args.proxy,
        outfile_prefix=args.outfile_prefix
    )
    print(f"Total execution time: {time.time() - start_time:.2f} seconds.")

[LLM] Loading tokenizer and model: meta-llama/Llama-3.1-8B-Instruct


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

[Pipeline] Loading dataset...

[Stage 1] Schema Understanding + LLM Checker

[Stage 2] Semantic Transformation + LLM Checker
[LLM] Transforming batch rows 0..19
[LLM] Transforming batch rows 20..39
[LLM] Transforming batch rows 40..59
[LLM] Transforming batch rows 60..79
[LLM] Transforming batch rows 80..99

[Stage 3] General Cleaning + LLM Checker
[LLM] Cleaning batch rows 0..19
[LLM] Cleaning batch rows 20..39
[LLM] Cleaning batch rows 40..59
[LLM] Cleaning batch rows 60..79
[LLM] Cleaning batch rows 80..99

[Stage 4] Feature Engineering + LLM Checker
[LLM] Feature-engineering batch rows 0..19
[LLM] Feature-engineering batch rows 20..39
[LLM] Feature-engineering batch rows 40..59
[LLM] Feature-engineering batch rows 60..79
[LLM] Feature-engineering batch rows 80..99

✅ Pipeline finished. Final report saved to results_final_report.json
{
    "config": {
        "model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "use_proxy": false,
        "sample_size": 100
    },
    "results"

In [5]:
# -------------------------
# Oracle Functions (Ground Truth)
# -------------------------

def oracle_transform(df: pd.DataFrame) -> pd.DataFrame:
    """Deterministic implementation of the business rule transformations."""
    df_out = df.copy()
    for col in df_out.select_dtypes(include=['object']).columns:
        df_out[col] = df_out[col].str.strip()
    
    if 'sex' in df_out.columns:
        df_out['sex'] = df_out['sex'].str.capitalize()
    
    if 'income' in df_out.columns:
        df_out['income'] = df_out['income'].astype(str).str.replace(r'\.', '', regex=True)
        df_out['income'] = np.where(df_out['income'].str.contains('>', na=False), '>50K', '<=50K')

    return df_out

def oracle_clean(df: pd.DataFrame) -> pd.DataFrame:
    """Deterministic implementation of the cleaning logic."""
    df_out = df.copy()
    for col in df_out.select_dtypes(include=np.number).columns:
        if df_out[col].isnull().any():
            median_val = df_out[col].median()
            df_out[col].fillna(median_val, inplace=True)
            
    for col in df_out.select_dtypes(include=['object']).columns:
         if df_out[col].isnull().any():
            df_out[col].fillna("Unknown", inplace=True)
            
    df_out.drop_duplicates(inplace=True)
    return df_out.reset_index(drop=True)


def oracle_features(df: pd.DataFrame) -> pd.DataFrame:
    """Deterministic implementation of feature engineering."""
    df_out = df.copy()
    df_out['is_high_earner'] = df_out['income'].apply(lambda x: 1 if isinstance(x, str) and '>50K' in x else 0).astype(int)
    
    hours = pd.to_numeric(df_out['hours-per-week'], errors='coerce')
    bins = [-1, 20, 40, 60, np.inf]
    labels = ['<=20', '21-40', '41-60', '>60']
    df_out['hours_bucket'] = pd.cut(hours, bins=bins, labels=labels, right=True).astype(str)
    
    return df_out

# -------------------------
# Helper Functions for Evaluation
# -------------------------

def load_and_parse_batches(directory: str, file_prefix: str) -> pd.DataFrame:
    """Finds all batch files for a stage, parses them, and combines the results."""
    batch_files = [f for f in os.listdir(directory) if f.startswith(file_prefix)]
    
    def get_batch_num(filename):
        match = re.search(r'_batch_(\d+)', filename)
        return int(match.group(1)) if match else -1
        
    batch_files.sort(key=get_batch_num)
    
    all_rows = []
    print(f"  > Found {len(batch_files)} files for prefix '{file_prefix}'...")
    for filename in batch_files:
        filepath = os.path.join(directory, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            
            parsed_json = extract_json_block(content)
            rows = parsed_json.get("rows", []) if isinstance(parsed_json, dict) else parsed_json if isinstance(parsed_json, list) else []
            all_rows.extend(rows)
        except Exception as e:
            print(f"    > Warning: Could not process file {filename}. Error: {e}")
            
    if not all_rows:
        print(f"    > Error: No rows could be extracted for prefix '{file_prefix}'. Returning empty DataFrame.")
        return pd.DataFrame()

    return pd.DataFrame(all_rows)

def compare_dataframes(df_llm: pd.DataFrame, df_oracle: pd.DataFrame) -> float:
    """Compares two dataframes cell-by-cell and returns an accuracy score."""
    if df_llm.empty or df_oracle.empty:
        return 0.0
        
    common_indices = df_llm.index.intersection(df_oracle.index)
    common_columns = df_llm.columns.intersection(df_oracle.columns)
    
    if len(common_indices) == 0 or len(common_columns) == 0:
        return 0.0

    df_llm_aligned = df_llm.loc[common_indices, common_columns].copy()
    df_oracle_aligned = df_oracle.loc[common_indices, common_columns].copy()

    comparison = df_llm_aligned.astype(str).fillna("NA") == df_oracle_aligned.astype(str).fillna("NA")
    
    total_cells = np.prod(comparison.shape)
    if total_cells == 0: return 1.0
        
    correct_cells = comparison.sum().sum()
    return correct_cells / total_cells

# -------------------------
# Main Evaluation Logic
# -------------------------
def evaluate_worker_files(results_dir="."):
    """Main function to run the evaluation from saved files."""
    print("--- Evaluating LLM Worker Performance from Saved Files ---")
    
    # 1. Recreate the initial state (the data with errors)
    print("\n[Step 1] Recreating initial data state...")
    df_full = load_uci_adult()
    if df_full.empty:
        print("Could not load data. Aborting.")
        return
    df_sample = df_full.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    df_with_errors = inject_errors(df_sample.copy(), {"missing_value":0.03, "swap_columns":0.02, "bad_casing":0.03, "stray_whitespace":0.03})
    print(f"Recreated {len(df_sample)} sample rows with errors.")

    # <-- ADDED SECTION START -->
    # 2. Evaluate Schema Understanding Worker
    print("\n[Step 2] Evaluating Schema Understanding LLM...")
    schema_accuracy = 0.0
    schema_file = os.path.join(results_dir, "results_raw_schema_0.txt")
    if os.path.exists(schema_file):
        try:
            with open(schema_file, 'r', encoding='utf-8') as f:
                content = f.read()
            
            parsed_schema = extract_json_block(content)
            llm_schema = parsed_schema.get("pred_schema", {})
            
            # Use the clean sample data to infer the true schema
            oracle_schema = infer_true_schema(df_sample)
            
            correct_cols = 0
            total_cols = len(oracle_schema)
            for col, correct_type in oracle_schema.items():
                if llm_schema.get(col) == correct_type:
                    correct_cols += 1
            
            if total_cols > 0:
                schema_accuracy = correct_cols / total_cols
            
        except Exception as e:
            print(f"  > Warning: Could not process schema file. Error: {e}")
    else:
        print(f"  > Warning: Schema file not found at '{schema_file}'. Skipping evaluation.")
    # <-- ADDED SECTION END -->

    # 3. Evaluate Semantic Transformation Worker
    print("\n[Step 3] Evaluating Semantic Transformation LLM...")
    df_llm_transformed = load_and_parse_batches(results_dir, "results_raw_transform_batch_")
    df_oracle_transformed = oracle_transform(df_with_errors)
    transform_accuracy = compare_dataframes(df_llm_transformed, df_oracle_transformed)

    # 4. Evaluate Data Cleaning Worker
    print("\n[Step 4] Evaluating Data Cleaning LLM...")
    df_llm_cleaned = load_and_parse_batches(results_dir, "results_raw_clean_batch_")
    df_oracle_cleaned = oracle_clean(df_oracle_transformed)
    clean_accuracy = compare_dataframes(df_llm_cleaned, df_oracle_cleaned)
    
    # 5. Evaluate Feature Engineering Worker
    print("\n[Step 5] Evaluating Feature Engineering LLM...")
    df_llm_features = load_and_parse_batches(results_dir, "results_raw_feature_batch_")
    df_oracle_features = oracle_features(df_oracle_cleaned)
    feature_accuracy = compare_dataframes(df_llm_features, df_oracle_features)
    
    # 6. Final Report
    print("\n--- LLM Worker Performance Report ---")
    print(f"Schema Understanding Accuracy : {schema_accuracy:.2%}") # <-- ADDED LINE
    print(f"Semantic Transformation Accuracy: {transform_accuracy:.2%}")
    print(f"Data Cleaning Accuracy          : {clean_accuracy:.2%}")
    print(f"Feature Engineering Accuracy  : {feature_accuracy:.2%}")
    print("-----------------------------------")

if __name__ == "__main__":
    evaluate_worker_files(results_dir=".")



--- Evaluating LLM Worker Performance from Saved Files ---

[Step 1] Recreating initial data state...
Recreated 100 sample rows with errors.

[Step 2] Evaluating Schema Understanding LLM...

[Step 3] Evaluating Semantic Transformation LLM...
  > Found 5 files for prefix 'results_raw_transform_batch_'...

[Step 4] Evaluating Data Cleaning LLM...
  > Found 5 files for prefix 'results_raw_clean_batch_'...

[Step 5] Evaluating Feature Engineering LLM...
  > Found 5 files for prefix 'results_raw_feature_batch_'...

--- LLM Worker Performance Report ---
Schema Understanding Accuracy : 100.00%
Semantic Transformation Accuracy: 94.73%
Data Cleaning Accuracy          : 95.47%
Feature Engineering Accuracy  : 94.35%
-----------------------------------


In [6]:
!zip -r archive.zip /kaggle/working/


  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/results_raw_feature_batch_40.txt (deflated 90%)
  adding: kaggle/working/results_raw_batched_checks_sex_map.txt (deflated 93%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/results_raw_feature_batch_80.txt (deflated 90%)
  adding: kaggle/working/results_raw_feature_batch_20.txt (deflated 91%)
  adding: kaggle/working/results_raw_feature_batch_0.txt (deflated 91%)
  adding: kaggle/working/results_raw_clean_batch_40.txt (deflated 87%)
  adding: kaggle/working/results_raw_batched_checks_trim_whitespace.txt (deflated 92%)
  adding: kaggle/working/results_raw_checker_cleaning.txt (deflated 87%)
  adding: kaggle/working/results_raw_clean_batch_60.txt (deflated 90%)
  adding: kaggle/working/results_raw_clean_batch_80.txt (deflated 87%)
  adding: kaggle/working/results_raw_batched_checks_income_map.txt (deflated 92%)
  adding: kaggle/working/results_raw_transform_batch_40_0.txt (deflated 90%)
  a

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:

import os
import re
import json
import pandas as pd
from typing import List, Dict, Tuple


In [9]:
# -------------------------------------
# Oracle Checkers (Ground Truth for Checkers)
# -------------------------------------

def oracle_check_transformation(row_before: pd.Series, row_after: pd.Series) -> Tuple[bool, str]:
    """Deterministic check for the semantic transformation rules."""
    # Check whitespace
    for col, val in row_after.items():
        if isinstance(val, str) and (val.startswith(' ') or val.endswith(' ')):
            return False, f"Whitespace not trimmed in column '{col}'"

    # Check sex capitalization
    if 'sex' in row_after and isinstance(row_after['sex'], str) and row_after['sex'] not in ["Male", "Female", "Unknown"]:
        return False, f"Sex '{row_after['sex']}' was not capitalized correctly"
    
    # Check income normalization
    if 'income' in row_after and isinstance(row_after['income'], str) and row_after['income'] not in ['>50K', '<=50K']:
         return False, f"Income '{row_after['income']}' was not normalized to '>50K' or '<=50K'"

    return True, "OK"

def oracle_check_cleaning(row_before: pd.Series, row_after: pd.Series) -> Tuple[bool, str]:
    """Deterministic check for the cleaning stage."""
    # Check if NaN values were handled
    if row_after.isnull().any().any():
        return False, "Row still contains NaN values after cleaning"
    
    # Heuristic: check if obvious placeholders from injection are gone
    if 'Unknown' in row_after.to_list() and pd.isna(row_before).any():
        # This is a weak check but confirms some imputation happened
        pass
    
    return True, "OK"

def oracle_check_features(row_before: pd.Series, row_after: pd.Series) -> Tuple[bool, str]:
    """Deterministic check for feature engineering."""
    # Check is_high_earner
    expected_high_earner = 1 if '>50K' in str(row_before.get('income')) else 0
    if int(row_after.get('is_high_earner', -1)) != expected_high_earner:
        return False, f"is_high_earner is {row_after.get('is_high_earner')}, expected {expected_high_earner}"
        
    # Check hours_bucket
    try:
        hours = float(row_before.get('hours-per-week', 0))
        if hours <= 20: expected_bucket = '<=20'
        elif hours <= 40: expected_bucket = '21-40'
        elif hours <= 60: expected_bucket = '41-60'
        else: expected_bucket = '>60'
        
        if row_after.get('hours_bucket') != expected_bucket:
             return False, f"hours_bucket is {row_after.get('hours_bucket')}, expected {expected_bucket} for {hours} hours"
    except (ValueError, TypeError):
         return False, f"Could not parse hours-per-week: {row_before.get('hours-per-week')}"

    return True, "OK"

# -------------------------------------
# Main Analysis Logic
# -------------------------------------

def analyze_checker_performance(results_dir="."):
    """Loads checker verdicts, compares to oracles, and prints a detailed report."""
    print("--- Analyzing LLM Checker Performance from Saved Files ---")

    # 1. Recreate all necessary DataFrames
    print("\n[Step 1] Recreating all data states from pipeline stages...")
    df_sample = load_uci_adult().sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    df_with_errors = inject_errors(df_sample.copy(), {"missing_value":0.03, "swap_columns":0.02, "bad_casing":0.03, "stray_whitespace":0.03})
    df_llm_transformed = load_and_parse_batches(results_dir, "results_raw_transform_batch_")
    df_llm_cleaned = load_and_parse_batches(results_dir, "results_raw_clean_batch_")
    df_llm_features = load_and_parse_batches(results_dir, "results_raw_feature_batch_")
    print("Data states recreated.")

    # 2. Analyze Transformation Checker
    print("\n--- Analysis: Semantic Transformation Checker ---")
    transform_files = [f for f in os.listdir(results_dir) if f.startswith("results_raw_batched_checks_")]
    all_transform_verdicts = []
    for f in transform_files:
        try:
            with open(os.path.join(results_dir, f), 'r', encoding='utf-8') as file:
                all_transform_verdicts.extend(extract_json_block(file.read()))
        except Exception as e:
            print(f"Warning: could not parse {f}: {e}")
    analyze_verdicts(all_transform_verdicts, df_with_errors, df_llm_transformed, oracle_check_transformation)

    # 3. Analyze Cleaning Checker
    print("\n--- Analysis: General Cleaning Checker ---")
    clean_verdicts = []
    try:
        with open(os.path.join(results_dir, "results_raw_checker_cleaning.txt"), 'r', encoding='utf-8') as file:
            clean_verdicts = extract_json_block(file.read())
    except Exception as e:
        print(f"Warning: could not parse cleaning checker file: {e}")
    analyze_verdicts(clean_verdicts, df_llm_transformed, df_llm_cleaned, oracle_check_cleaning)
    
    # 4. Analyze Feature Engineering Checker
    print("\n--- Analysis: Feature Engineering Checker ---")
    feature_files = [f for f in os.listdir(results_dir) if f.startswith("results_raw_checker_feature_")]
    all_feature_verdicts = []
    for f in feature_files:
        try:
            with open(os.path.join(results_dir, f), 'r', encoding='utf-8') as file:
                all_feature_verdicts.extend(extract_json_block(file.read()))
        except Exception as e:
            print(f"Warning: could not parse {f}: {e}")
    analyze_verdicts(all_feature_verdicts, df_llm_cleaned, df_llm_features, oracle_check_features)

def analyze_verdicts(llm_verdicts: List[Dict], df_before: pd.DataFrame, df_after: pd.DataFrame, oracle_checker_func):
    """Generic function to compare LLM verdicts to an oracle checker."""
    if not llm_verdicts:
        print("No verdicts found to analyze.")
        return

    tp, tn, fp, fn = 0, 0, 0, 0
    fp_examples, fn_examples = [], []

    # Use a set to only check each unique row index once
    checked_indices = set()

    for verdict in llm_verdicts:
        try:
            idx = int(verdict.get("row_idx"))
            if idx in checked_indices:
                continue
            checked_indices.add(idx)

            llm_ok = verdict.get("verdict", "").upper() == "OK"
            oracle_ok, oracle_reason = oracle_checker_func(df_before.iloc[idx], df_after.iloc[idx])

            if llm_ok and oracle_ok:
                tp += 1
            elif not llm_ok and not oracle_ok:
                tn += 1
            elif llm_ok and not oracle_ok:
                fp += 1
                fp_examples.append(f"  - Row {idx}: Checker said OK, but Oracle failed. Reason: {oracle_reason}")
            elif not llm_ok and oracle_ok:
                fn += 1
                fn_examples.append(f"  - Row {idx}: Checker said NOT_OK (Notes: '{verdict.get('notes', 'N/A')}'), but Oracle found no error.")
        except (IndexError, TypeError, ValueError) as e:
            print(f"  > Warning: Could not process verdict: {verdict}. Reason: {e}")

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total > 0 else 0
    print(f"Checker Performance ({total} verdicts analyzed):")
    print(f"  - Accuracy: {accuracy:.2%}")
    print(f"  - True Positives (Correct OKs)    : {tp}")
    print(f"  - True Negatives (Correct NOT_OKs): {tn}")
    print(f"  - False Positives (Missed Errors) : {fp}")
    print(f"  - False Negatives (Wrongly Flagged): {fn}")

    if fp_examples:
        print("\n  Examples of False Positives (Checker Missed These Errors):")
        for ex in fp_examples[:3]: # Print first 3 examples
            print(ex)
    if fn_examples:
        print("\n  Examples of False Negatives (Checker Flagged Correct Rows):")
        for ex in fn_examples[:3]: # Print first 3 examples
            print(ex)

if __name__ == "__main__":
    analyze_checker_performance(results_dir=".")


--- Analyzing LLM Checker Performance from Saved Files ---

[Step 1] Recreating all data states from pipeline stages...
  > Found 5 files for prefix 'results_raw_transform_batch_'...
  > Found 5 files for prefix 'results_raw_clean_batch_'...
  > Found 5 files for prefix 'results_raw_feature_batch_'...
Data states recreated.

--- Analysis: Semantic Transformation Checker ---
Checker Performance (30 verdicts analyzed):
  - Accuracy: 56.67%
  - True Positives (Correct OKs)    : 16
  - True Negatives (Correct NOT_OKs): 1
  - False Positives (Missed Errors) : 3
  - False Negatives (Wrongly Flagged): 10

  Examples of False Positives (Checker Missed These Errors):
  - Row 53: Checker said OK, but Oracle failed. Reason: Income '<=50K.' was not normalized to '>50K' or '<=50K'
  - Row 45: Checker said OK, but Oracle failed. Reason: Income '<=50K.' was not normalized to '>50K' or '<=50K'
  - Row 55: Checker said OK, but Oracle failed. Reason: Income '<=50K.' was not normalized to '>50K' or '<=50